# Survey Processing Contamination Detection

This lab demonstrates a local, traditional ML approach to a targeted survey data quality problem. The goal is not to reach for an LLM, but to show how rules, dictionaries, and standard Python ML tools can solve the problem privately and reproducibly.

## Setup

The notebook adds the local `src` directory to `sys.path` so it can run from the project root or from inside the `notebooks` folder. All processing stays on this machine, which matters when survey records contain respondent or business data that should not be sent to an external service.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from survey_contamination.data import load_first_names, load_surnames
from survey_contamination.detectors import (
    BusinessSuffixHeuristicDetector,
    FeatureClassifierDetector,
    NameDictionaryDetector,
    build_name_sets,
)
from survey_contamination.evaluation import evaluate_detector_predictions, summarize_error_examples
from survey_contamination.synthetic import generate_labeled_business_names

## Load Public Name Data

The detectors use public first-name and surname frequency tables as dictionaries. This is intentionally simple: for a narrow contamination problem, transparent local reference data is often enough to build a useful first pass.

In [ ]:
surnames = load_surnames(top_n=5000)
first_names = load_first_names(top_n=5000)
surnames.head(), first_names.head()

## Generate a Low-Rate Contamination Sample

The simulated survey file has a 0.5% contamination rate. At this base rate, false positives are operationally expensive: even a detector with reasonable precision can create many records for manual review if it flags too aggressively.

In [ ]:
frame = generate_labeled_business_names(first_names, surnames, n_records=10000, contamination_rate=0.005, seed=42)
frame["is_contaminated"].value_counts()

## Rule-Based Detectors

The first two detectors are local rule systems: one scores dictionary name matches, and the other reduces scores for names that also contain ordinary business suffixes. These rules are inspectable and reproducible, which makes them a good fit for sensitive data-quality workflows.

In [ ]:
first_lookup, surname_lookup = build_name_sets(first_names, surnames)
detectors = [
    NameDictionaryDetector(first_lookup, surname_lookup),
    BusinessSuffixHeuristicDetector(first_lookup, surname_lookup),
]

## Traditional ML Baseline

The feature classifier is a traditional machine-learning baseline built from engineered features and a scikit-learn model. It is not an LLM: it does not call an external model, generate text, or require sending survey data outside the local environment.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

train, test = train_test_split(frame, test_size=0.35, random_state=42, stratify=frame["is_contaminated"])
classifier = FeatureClassifierDetector(first_lookup, surname_lookup)
classifier.fit(train["business_name"], train["is_contaminated"])
detectors.append(classifier)

scored = []
for detector in detectors:
    output = test.copy()
    output["detector"] = detector.name
    output["score"] = detector.score(output["business_name"])
    output["predicted"] = detector.predict(output["business_name"])
    scored.append(output)
scores = pd.concat(scored, ignore_index=True)

## Compare Detector Metrics

Precision, recall, and F1 summarize different operational trade-offs. In a low-contamination setting, precision deserves special attention because each false positive can become a costly review task for analysts or survey operations teams.

In [ ]:
metrics = pd.DataFrame(
    evaluate_detector_predictions(group, detector_name=name)
    for name, group in scores.groupby("detector")
).sort_values("f1", ascending=False)
metrics

## Inspect Errors

Aggregate metrics are not enough for a production workflow. Reviewing false positives and false negatives helps decide whether a detector should be tuned for automated filtering, triage, or analyst review.

In [ ]:
errors = pd.concat(
    summarize_error_examples(group.assign(detector=name), max_examples=5)
    for name, group in scores.groupby("detector")
)
errors[["detector", "error_type", "business_name", "is_contaminated", "predicted", "score"]]

## Optional Extensions

spaCy can be explored as an optional local NLP extension for person-name recognition, but the core lab does not require it. The main lesson is that dictionaries, rules, feature engineering, and traditional Python ML can solve this targeted problem privately and reproducibly before reaching for heavier tools.